Para este juego de Battleship crearemos los siguientes objetos:
1. Player
2. Ship
3. Board
4. Game

In [ ]:
from google.colab import output

In [ ]:
class Ship():
  def __init__(self, name, length, alias):
    self.name = name
    self.length = length
    self.alias = alias

  def __str__(self):
    return f"{self.name}[{self.length}]"

  def __repr__(self):
    return str(self)

class Battleship(Ship):
  def __init__(self):
    super().__init__("Battleship", 4, 1)

class Carrier(Ship):
  def __init__(self):
    super().__init__("Carrier", 5, 2)

class Destroyer(Ship):
  def __init__(self):
    super().__init__("Destroyer", 2, 3)

class Submarine(Ship):
  def __init__(self):
    super().__init__("Submarine", 3, 4)

class Cruiser(Ship):
  def __init__(self):
    super().__init__("Cruiser", 3, 5)

In [ ]:
import numpy as np
class Board():

  pretty_print_mappings = {}

  def __init__(self, n=10):
    self.n = n
    self.board = np.zeros((n,n))

  def _is_in_board_range(self, x, y):
    return 0 <= x < self.n and 0 <= y < self.n

  def __repr__(self):
    return str(self)

  def __str__(self):
    formatted_board = self.board.astype(str).copy()
    for key, value in self.pretty_print_mappings.items():
      formatted_board[formatted_board == key] = value
    board_str = self.matrix_to_str(formatted_board)
    return board_str

  @staticmethod
  def matrix_to_str(mtx):
    str_matrix = ""
    for i in range(mtx.shape[0]):
      row = ""
      for j in range(mtx.shape[1]):
        row += str(mtx[i,j])
      str_matrix += row + "\n"
    return str_matrix

class BombBoard(Board):
  """
    FOR THE BOMB BOARDS
    - 0 means no bomb has been placed.
    - 1 means a bomb succesfully hit something.
    - 2 means a bomb hit nothing.
  """

  pretty_print_mappings = {
      "0.0": "▫️",
      "1.0": "🔴",
      "2.0": "⚫"
  }

  def place_bomb(self, x, y, oponent_board):
    # 1. Comprobar que las coordenadas de la bomba están dentro del tablero.
    if(not(self._is_in_board_range(x, y))):
      return False
    # 2. Comprobar que no hay otra bomba ya puesta.
    if(self.board[x,y] != 0):
      return False
    hit = oponent_board.check_if_ship_is_placed(x,y)
    self.board[x,y] = 1 if hit else 2
    return True

class ShipBoard(Board):
  """
    FOR THE SHIP BOARDS
    - 0 means no ship piece is placed there.
    - Anything else means a ship piece is there.
  """

  pretty_print_mappings = {
      "0.0": "▫️",
      "1.0": "🔒",
      "2.0": "🩸",
      "3.0": "🔥",
      "4.0": "⚡",
      "5.0": "♠️",
  }

  def __place_single_ship_piece(self, x, y, alias):
    self.board[x,y] = alias

  def place_ship(self, ship, x, y, orientation="h"):

    assert orientation in ["h", "v"]

    # 1. Comprobar que x,y esta dentro del tablero
    if not(self._is_in_board_range(x, y)):
      return False

    # Siempre para colocar si hace de izq. a der. y de top a down en el tablero
    if orientation == "v":
      x1, x2 = x, x+ship.length-1
      y1, y2 = y, y
    else:
      x1, x2 = x, x
      y1, y2 = y, y+ship.length-1

    # 2. Comprobar si cabe
    if not(self._is_in_board_range(x2, y2)):
      return False

    # 3. Comprobar si choca con otro barco
    if orientation == "v" and not(self.board[x1:x2+1, y].sum() == 0):
      return False
    elif orientation == "h" and not(self.board[x, y1:y2+1].sum() == 0):
      return False

    # 4. Colocarlo pieza a pieza usando __place_single_ship_piece
    for i in range(ship.length):
      if orientation == "v":
        self.__place_single_ship_piece(x+i, y, ship.alias)
      if orientation == "h":
        self.__place_single_ship_piece(x, y+i, ship.alias)

    # Devuelve si se pudo colocar finalmente
    return True

  def check_if_ship_is_placed(self, x, y):
    return self.board[x,y] != 0

In [ ]:
import random
class Player():

  ship_options = [Carrier, Battleship, Submarine, Cruiser, Destroyer]

  def __init__(self, name):
    self.name = name
    self.ships = []
    self.bomb_board = BombBoard()
    self.ship_board = ShipBoard()
    self.__create_ships()

  def _get_coord_input(self):
    raise NotImplementedError

  def _get_orientation_input(self):
    raise NotImplementedError

  def __create_ships(self):
    for ship in self.ship_options:
      self.ships.append(ship())

  def _find_ship_by_alias(self, alias):
    for ship in self.ships:
      if ship.alias == alias:
        return ship
    return None

  def initialize_ships(self):
    print(f"Setting player {self.name} ships positions")
    for ship in self.ships:
      placed = False
      while not(placed):
        print(f"Colocando {ship}...")
        print(self.ship_board)
        try:
          x, y = self._get_coord_input()
          orientation = self._get_orientation_input()
        except(ValueError, AssertionError):
          output.clear()
          print("Inputs inválidos.")
          continue
        output.clear()
        placed = self.ship_board.place_ship(ship, x, y, orientation)
        if not placed: print("Intente otra vez, posición de barco inválida.")

  def get_number_of_succesful_hits(self):
    return np.count_nonzero(self.bomb_board.board == 1.0)

  def get_number_of_ship_pieces(self):
    n = self.ship_board.n
    return n*n - np.count_nonzero(self.ship_board.board == 0.0)

  def place_bomb(self, ref_board):
    while True:
      print(f"Jugador {self.name}, indica en qué coordenada deseas colocar una bomba:")
      print(self.bomb_board)
      try:
        x, y = self._get_coord_input()
        assert self.bomb_board.place_bomb(x, y, ref_board)
        hit_ship = self._find_ship_by_alias(ref_board.board[x,y])
        output.clear()
        if hit_ship != None: print(f"Hit {hit_ship}!")
        return
      except (AssertionError, ValueError):
        output.clear()
        print("Coordenadas inválidas. Intente de nuevo.")

class HumanPlayer(Player):

  def _get_coord_input(self):
    coord = input("(x,y)=").split(",")
    coord = list(map(int, coord))
    assert len(coord) == 2
    x, y = coord[0], coord[1]
    return x, y

  def _get_orientation_input(self):
    orientation = input("Orientation (h/v)=")
    assert orientation in ["h", "v"]
    return orientation

class BotPlayer(Player):

  def _get_coord_input(self):
    x = random.randint(0,self.ship_board.n-1)
    y = random.randint(0,self.ship_board.n-1)
    return x, y

  def _get_orientation_input(self):
    return random.choice(["h", "v"])


In [ ]:
from google.colab import output

class Game():

  n_players = 2

  def __init__(self, p1_name, p2_name):
    self.players = [HumanPlayer(p1_name), BotPlayer(p2_name)]
    self.turn = 0

  def start_game(self):
    for player in self.players:
      player.initialize_ships()
      if isinstance(player, HumanPlayer):
        print(player.ship_board)
        input("Listo! Press to continue..")
    self.__main_loop()

  @staticmethod
  def show_game_info(player):
    print(f"Player {player.name} turn")
    print(player.bomb_board)

  def __main_loop(self):
    while True:
      # Print game status
      curr_player = self.players[self.turn%self.n_players]
      oponent_player = self.players[(self.turn+1)%self.n_players]
      output.clear()
      # 1. Pedir al jugador en turno que coloque su bomba.
      curr_player.place_bomb(oponent_player.ship_board)
      self.show_game_info(curr_player)
      input("Press to continue..")
      # 2. Verificar si se acabo el juego.
      if curr_player.get_number_of_succesful_hits() ==  oponent_player.get_number_of_ship_pieces():
        print(f"Player {curr_player.name} won!")
        return
      self.turn += 1

In [ ]:
game = Game("Juan", "Pedro")
game.start_game()

Jugador Juan, indica en qué coordenada deseas colocar una bomba:
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️
▫️▫️▫️▫️▫️▫️▫️▫️▫️▫️

